# PravahX — DualSPHysics Google Colab GPU Runner

This notebook runs 3D SPH dam break and surge simulations on a **Google Colab GPU (NVIDIA Tesla T4 / V100 / A100)**.

### Workflow:
1. Verify GPU acceleration (`nvidia-smi`).
2. Download and set up DualSPHysics Linux x64 CUDA binaries.
3. Upload your prepared `dualsphysics_case.tar.gz` from PravahX.
4. Execute the simulation and generate cryptographic SHA-256 manifest.
5. Download `dualsphysics_results.tar.gz` for import into PravahX.


In [ ]:
# Step 1: Check GPU Acceleration
import subprocess

try:
    res = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
    if res.returncode == 0:
        print(res.stdout)
        print('✅ NVIDIA GPU is active and ready.')
    else:
        print('⚠️ WARNING: nvidia-smi failed:')
        print(res.stderr)
        print('Please enable GPU: Colab menu -> Runtime ->')
        print('  Change runtime type -> T4 GPU -> Save.')
except Exception as e:
    print(f'⚠️ Could not execute nvidia-smi: {e}')
    print('Please enable GPU: Colab menu -> Runtime ->')
    print('  Change runtime type -> T4 GPU -> Save.')


In [ ]:
# Step 2: Download & Setup DualSPHysics Linux Binaries
import os
import shutil
import subprocess
import urllib.request

bin_dir = '/content/dualsphysics/bin'
os.makedirs(bin_dir, exist_ok=True)

base_url = 'https://dual.sphysics.org/sphcourse/DualSPHysics-bin/dualsphysics/bin'
fb_base = 'https://raw.githubusercontent.com/DualSPHysics/DualSPHysics/master/bin/linux'

files_to_download = [
    'GenCase_linux64',
    'DualSPHysics5.4_linux64',
    'PartVTK_linux64',
    'libChronoEngine.so',
    'libdsphchrono.so',
    'DsphConfig.xml',
]

print('Downloading DualSPHysics Linux x64 binaries...')
for fname in files_to_download:
    dest = os.path.join(bin_dir, fname)
    if os.path.exists(dest) and os.path.getsize(dest) > 0:
        print(f'  [cached] {fname}')
        continue

    url_primary = f'{base_url}/{fname}'
    success = False
    for url in [url_primary, f'{fb_base}/{fname}']:
        try:
            print(f'  Downloading {fname} from {url[:40]}...')
            hdr = {'User-Agent': 'Mozilla/5.0'}
            req = urllib.request.Request(url, headers=hdr)
            with (
                urllib.request.urlopen(req, timeout=60) as resp,
                open(dest, 'wb') as out_f,
            ):
                shutil.copyfileobj(resp, out_f)
            success = True
            break
        except Exception as e:
            print(f'    Failed from {url[:40]}: {e}')

    if not success:
        raise RuntimeError(
            f'Failed to download required DualSPHysics file: {fname}'
        )

    if fname.endswith('_linux64'):
        os.chmod(dest, 0o755)

# Install shared libraries to /usr/local/lib and update cache
for lib in ['libChronoEngine.so', 'libdsphchrono.so']:
    src = os.path.join(bin_dir, lib)
    if os.path.exists(src):
        shutil.copy2(src, f'/usr/local/lib/{lib}')
subprocess.run(['ldconfig'], check=False)

# Create symlinks in /usr/local/bin for canonical names and aliases
aliases = {
    'GenCase_linux64': ['GenCase_linux64', 'gencase'],
    'DualSPHysics5.4_linux64': [
        'DualSPHysics5.4_linux64',
        'dualsphysics_gpu',
        'dualsphysics',
    ],
    'PartVTK_linux64': ['PartVTK_linux64', 'partvtk'],
}

for target_file, links in aliases.items():
    target_path = os.path.join(bin_dir, target_file)
    for link_name in links:
        link_path = f'/usr/local/bin/{link_name}'
        try:
            if os.path.islink(link_path) or os.path.exists(link_path):
                os.remove(link_path)
            os.symlink(target_path, link_path)
        except Exception as e:
            print(f'Could not symlink {link_name}: {e}')

print('\n--- Verifying DualSPHysics Installation ---')
!gencase -ver
!dualsphysics_gpu -ver


In [ ]:
# Step 3: Upload Case Package
import os
import tarfile

from google.colab import files

run_dir = '/content/run'
os.makedirs(run_dir, exist_ok=True)

# Check if user uploaded dualsphysics_case.tar.gz via sidebar
tar_candidates = [
    '/content/dualsphysics_case.tar.gz',
    '/content/run/dualsphysics_case.tar.gz',
]
tar_candidate = next(
    (c for c in tar_candidates if os.path.exists(c)), None
)

if not tar_candidate:
    print('Upload your dualsphysics_case.tar.gz prepared by PravahX:')
    uploaded = files.upload()
    for fn in uploaded:
        if fn.endswith('.tar.gz') or fn.endswith('.tgz'):
            tar_candidate = fn
            break

if not tar_candidate or not os.path.exists(tar_candidate):
    raise FileNotFoundError(
        'No dualsphysics_case.tar.gz uploaded.'
    )

print(f'Extracting {tar_candidate} into {run_dir}...')
with tarfile.open(tar_candidate, 'r:gz') as tar:
    if hasattr(tarfile, 'data_filter'):
        tar.extractall(path=run_dir, filter='data')
    else:
        tar.extractall(path=run_dir)

print('Extracted files in /content/run:')
for f in os.listdir(run_dir):
    sz = os.path.getsize(os.path.join(run_dir, f))
    print(f'  - {f} ({sz} bytes)')


In [ ]:
# Step 4: Execute Simulation on GPU
%%bash
cd /content/run
chmod +x run_colab.sh
./run_colab.sh


In [ ]:
# Step 5: Download Verified Results Package
import os

from google.colab import files

results_path = '/content/run/dualsphysics_results.tar.gz'
if os.path.exists(results_path):
    size_mb = os.path.getsize(results_path) / (1024 * 1024)
    print(
        f'Downloading results ({size_mb:.2f} MB) for PravahX import...'
    )
    files.download(results_path)
else:
    print('Error: dualsphysics_results.tar.gz not found.')
    print('Check the output of Step 4 for simulation errors.')
